# Temporal modeling experiment

Compare the persistence baseline, a financial Ridge logistic regression, and a financial + sporting Ridge logistic regression using expanding-window temporal validation.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

df = pd.read_csv('../data/processed/modeling_data_v1.csv')

## Define features and common comparison sample

In [2]:
financial_features = [
    'current_liquidity',
    'liability_ratio',
    'return_on_assets',
]

sporting_features = [
    'division',
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
    'promoted',
    'relegated',
]

common_sample = df.dropna(subset=sporting_features).copy()

common_sample.shape

(297, 34)

In [3]:
common_sample.groupby('fiscal_year')['loss_next_year'].agg(
    observations='count',
    losses='sum',
    loss_rate='mean',
)

,observations,losses,loss_rate
fiscal_year,,,
2016,31,18,0.580645
2017,32,16,0.500000
2018,32,17,0.531250
2019,33,24,0.727273
2020,35,17,0.485714
2021,35,15,0.428571
2022,35,14,0.400000
2023,33,24,0.727273
2024,31,20,0.645161


## Model specification

Continuous variables are standardized inside each training fold. `division` is one-hot encoded with Série A as the reference category. `promoted` and `relegated` remain binary.

In [4]:
financial_continuous = financial_features

full_continuous = financial_features + [
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
]

full_categorical = ['division']
full_binary = ['promoted', 'relegated']

C_GRID = [0.01, 0.1, 1, 10, 100]

In [5]:
def build_pipeline(model_type, C):
    if model_type == 'financial':
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', StandardScaler(), financial_continuous),
            ],
            remainder='drop',
        )
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', StandardScaler(), full_continuous),
                (
                    'division',
                    OneHotEncoder(
                        categories=[[1, 2, 3, 4]],
                        drop='first',
                        handle_unknown='ignore',
                    ),
                    full_categorical,
                ),
                ('binary', 'passthrough', full_binary),
            ],
            remainder='drop',
        )

    return Pipeline([
        ('preprocess', preprocessor),
        (
            'model',
            LogisticRegression(
                C=C,
                solver='liblinear',
                max_iter=1000,
            ),
        ),
    ])

## Temporal selection of C

For each external fold, `C` is selected using only earlier feature years. Internal validation also follows an expanding window. ROC-AUC is calculated separately in each internal validation year, and `C` is selected by the mean of those temporal AUCs. In an exact tie, the smaller `C` is chosen.

In [6]:
def select_c(train_data, model_type):
    years = sorted(train_data['fiscal_year'].unique())
    tuning_rows = []

    for C in C_GRID:
        fold_aucs = []
        validation_years = []

        for validation_year in years:
            inner_train = train_data[
                train_data['fiscal_year'] < validation_year
            ]
            inner_validation = train_data[
                train_data['fiscal_year'] == validation_year
            ]

            if inner_train['fiscal_year'].nunique() < 2:
                continue

            if inner_train['loss_next_year'].nunique() < 2:
                continue

            if inner_validation['loss_next_year'].nunique() < 2:
                continue

            model = build_pipeline(model_type, C)
            model.fit(inner_train, inner_train['loss_next_year'])

            probabilities = model.predict_proba(inner_validation)[:, 1]
            fold_auc = roc_auc_score(
                inner_validation['loss_next_year'],
                probabilities,
            )

            fold_aucs.append(fold_auc)
            validation_years.append(validation_year)

        tuning_rows.append({
            'C': C,
            'mean_inner_roc_auc': np.mean(fold_aucs),
            'std_inner_roc_auc': np.std(fold_aucs, ddof=0),
            'validation_years': validation_years,
        })

    tuning = pd.DataFrame(tuning_rows)
    tuning = tuning.sort_values(
        ['mean_inner_roc_auc', 'C'],
        ascending=[False, True],
    ).reset_index(drop=True)

    return tuning.loc[0, 'C'], tuning

## Expanding-window external experiment

The external test folds are defined by feature year. Therefore `fiscal_year = 2020` predicts the financial outcome in `target_year = 2021`, and so on.

In [7]:
def run_experiment(data, first_test_year):
    prediction_rows = []
    tuning_rows = []

    test_years = [
        year
        for year in sorted(data['fiscal_year'].unique())
        if year >= first_test_year
    ]

    for test_year in test_years:
        train = data[data['fiscal_year'] < test_year].copy()
        test = data[data['fiscal_year'] == test_year].copy()

        baseline_predictions = test['loss_current_year'].astype(int)

        for row_index, prediction in zip(test.index, baseline_predictions):
            prediction_rows.append({
                'club': test.loc[row_index, 'club'],
                'fiscal_year': test_year,
                'target_year': test.loc[row_index, 'target_year'],
                'model': 'baseline',
                'y_true': int(test.loc[row_index, 'loss_next_year']),
                'score': float(prediction),
                'prediction': int(prediction),
            })

        for model_type, model_name in [
            ('financial', 'financial_ridge'),
            ('full', 'financial_sporting_ridge'),
        ]:
            best_C, tuning = select_c(train, model_type)

            for _, tuning_row in tuning.iterrows():
                tuning_rows.append({
                    'test_feature_year': test_year,
                    'model': model_name,
                    'C': tuning_row['C'],
                    'mean_inner_roc_auc': tuning_row['mean_inner_roc_auc'],
                    'std_inner_roc_auc': tuning_row['std_inner_roc_auc'],
                    'selected': tuning_row['C'] == best_C,
                    'validation_years': tuning_row['validation_years'],
                })

            model = build_pipeline(model_type, best_C)
            model.fit(train, train['loss_next_year'])

            probabilities = model.predict_proba(test)[:, 1]
            predictions = (probabilities >= 0.5).astype(int)

            for row_index, probability, prediction in zip(
                test.index,
                probabilities,
                predictions,
            ):
                prediction_rows.append({
                    'club': test.loc[row_index, 'club'],
                    'fiscal_year': test_year,
                    'target_year': test.loc[row_index, 'target_year'],
                    'model': model_name,
                    'y_true': int(test.loc[row_index, 'loss_next_year']),
                    'score': float(probability),
                    'prediction': int(prediction),
                })

    predictions = pd.DataFrame(prediction_rows)
    tuning_results = pd.DataFrame(tuning_rows)

    return predictions, tuning_results

## Main experiment

In [8]:
predictions, tuning_results = run_experiment(
    common_sample,
    first_test_year=2020,
)

In [9]:
tuning_results[tuning_results['selected']].reset_index(drop=True)

,test_feature_year,model,C,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2020,financial_ridge,0.01,0.738453,0.020806,True,"[2018, 2019]"
1,2020,financial_sporting_ridge,0.10,0.692021,0.104058,True,"[2018, 2019]"
2,2021,financial_ridge,0.10,0.670461,0.084554,True,"[2018, 2019, 2020]"
3,2021,financial_sporting_ridge,0.01,0.627723,0.126948,True,"[2018, 2019, 2020]"
4,2022,financial_ridge,0.10,0.641179,0.089075,True,"[2018, 2019, 2020, 2021]"
5,2022,financial_sporting_ridge,0.10,0.666027,0.127895,True,"[2018, 2019, 2020, 2021]"
6,2023,financial_ridge,0.10,0.628590,0.083555,True,"[2018, 2019, 2020, 2021, 2022]"
7,2023,financial_sporting_ridge,0.10,0.632141,0.132961,True,"[2018, 2019, 2020, 2021, 2022]"
8,2024,financial_ridge,0.10,0.617189,0.080422,True,"[2018, 2019, 2020, 2021, 2022, 2023]"
9,2024,financial_sporting_ridge,0.10,0.644068,0.124272,True,"[2018, 2019, 2020, 2021, 2022, 2023]"


## Pooled out-of-time performance — complementary

These metrics concatenate predictions from different external folds. They are retained as complementary diagnostics, not as the principal ROC-AUC summary.

In [10]:
pooled_results = (
    predictions
    .groupby('model')
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'pooled_roc_auc': roc_auc_score(group['y_true'], group['score']),
            'pooled_balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'pooled_accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

pooled_results

,model,observations,pooled_roc_auc,pooled_balanced_accuracy,pooled_accuracy
0,baseline,169.0,0.630802,0.630802,0.633136
1,financial_ridge,169.0,0.499578,0.460970,0.485207
2,financial_sporting_ridge,169.0,0.580450,0.586639,0.597633


## Performance by external feature year

In [11]:
yearly_results = (
    predictions
    .groupby(['fiscal_year', 'model'])
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'roc_auc': roc_auc_score(group['y_true'], group['score']),
            'balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

yearly_results

,fiscal_year,model,observations,roc_auc,balanced_accuracy,accuracy
0,2020,baseline,35.0,0.549020,0.549020,0.542857
1,2020,financial_ridge,35.0,0.532680,0.467320,0.457143
2,2020,financial_sporting_ridge,35.0,0.496732,0.578431,0.571429
3,2021,baseline,35.0,0.600000,0.600000,0.600000
4,2021,financial_ridge,35.0,0.553333,0.433333,0.371429
5,2021,financial_sporting_ridge,35.0,0.740000,0.583333,0.542857
6,2022,baseline,35.0,0.738095,0.738095,0.742857
7,2022,financial_ridge,35.0,0.578231,0.488095,0.400000
8,2022,financial_sporting_ridge,35.0,0.496599,0.559524,0.542857
9,2023,baseline,33.0,0.618056,0.618056,0.545455


## Principal temporal ROC-AUC summary

The principal ROC-AUC result is the simple mean of the external yearly AUCs. Each test year receives equal weight.

In [12]:
temporal_auc_results = (
    yearly_results
    .groupby('model')['roc_auc']
    .agg(
        mean_roc_auc='mean',
        std_roc_auc='std',
        min_roc_auc='min',
        max_roc_auc='max',
        folds='count',
    )
    .reset_index()
)

temporal_auc_results

,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,baseline,0.640580,0.076342,0.549020,0.738095,5
1,financial_ridge,0.568522,0.032192,0.532680,0.618182,5
2,financial_sporting_ridge,0.612861,0.113608,0.496599,0.740000,5


## Sensitivity analysis without feature year 2020

Only rows whose feature year is 2020 are removed. Earlier rows whose target happens to be 2020 remain valid.

In [13]:
sample_without_2020 = common_sample[
    common_sample['fiscal_year'] != 2020
].copy()

predictions_without_2020, tuning_without_2020 = run_experiment(
    sample_without_2020,
    first_test_year=2021,
)

In [14]:
yearly_results_without_2020 = (
    predictions_without_2020
    .groupby(['fiscal_year', 'model'])
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'roc_auc': roc_auc_score(group['y_true'], group['score']),
            'balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

yearly_results_without_2020

,fiscal_year,model,observations,roc_auc,balanced_accuracy,accuracy
0,2021,baseline,35.0,0.600000,0.600000,0.600000
1,2021,financial_ridge,35.0,0.510000,0.491667,0.457143
2,2021,financial_sporting_ridge,35.0,0.630000,0.541667,0.514286
3,2022,baseline,35.0,0.738095,0.738095,0.742857
4,2022,financial_ridge,35.0,0.690476,0.523810,0.428571
5,2022,financial_sporting_ridge,35.0,0.482993,0.511905,0.514286
6,2023,baseline,33.0,0.618056,0.618056,0.545455
7,2023,financial_ridge,33.0,0.472222,0.361111,0.424242
8,2023,financial_sporting_ridge,33.0,0.731481,0.756944,0.696970
9,2024,baseline,31.0,0.697727,0.697727,0.741935


In [15]:
temporal_auc_results_without_2020 = (
    yearly_results_without_2020
    .groupby('model')['roc_auc']
    .agg(
        mean_roc_auc='mean',
        std_roc_auc='std',
        min_roc_auc='min',
        max_roc_auc='max',
        folds='count',
    )
    .reset_index()
)

temporal_auc_results_without_2020

,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,baseline,0.663470,0.065405,0.600000,0.738095,4
1,financial_ridge,0.576129,0.102219,0.472222,0.690476,4
2,financial_sporting_ridge,0.596346,0.108499,0.482993,0.731481,4


In [16]:
pooled_results_without_2020 = (
    predictions_without_2020
    .groupby('model')
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'pooled_roc_auc': roc_auc_score(group['y_true'], group['score']),
            'pooled_balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'pooled_accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

pooled_results_without_2020

,model,observations,pooled_roc_auc,pooled_balanced_accuracy,pooled_accuracy
0,baseline,134.0,0.657983,0.657983,0.656716
1,financial_ridge,134.0,0.512014,0.455985,0.485075
2,financial_sporting_ridge,134.0,0.557826,0.569167,0.574627


In [17]:
tuning_without_2020[
    tuning_without_2020['selected']
].reset_index(drop=True)

,test_feature_year,model,C,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2021,financial_ridge,0.01,0.738453,0.020806,True,"[2018, 2019]"
1,2021,financial_sporting_ridge,0.10,0.692021,0.104058,True,"[2018, 2019]"
2,2022,financial_ridge,0.01,0.662302,0.109026,True,"[2018, 2019, 2021]"
3,2022,financial_sporting_ridge,0.01,0.675566,0.097007,True,"[2018, 2019, 2021]"
4,2023,financial_ridge,0.01,0.669346,0.095204,True,"[2018, 2019, 2021, 2022]"
5,2023,financial_sporting_ridge,0.01,0.627423,0.118369,True,"[2018, 2019, 2021, 2022]"
6,2024,financial_ridge,0.01,0.629921,0.116053,True,"[2018, 2019, 2021, 2022, 2023]"
7,2024,financial_sporting_ridge,0.01,0.648235,0.113760,True,"[2018, 2019, 2021, 2022, 2023]"


## Compare principal main and sensitivity results

In [18]:
comparison = pd.concat([
    temporal_auc_results.assign(sample='main'),
    temporal_auc_results_without_2020.assign(sample='without_2020'),
], ignore_index=True)

comparison[
    ['sample', 'model', 'mean_roc_auc', 'std_roc_auc', 'min_roc_auc', 'max_roc_auc', 'folds']
]

,sample,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,main,baseline,0.640580,0.076342,0.549020,0.738095,5
1,main,financial_ridge,0.568522,0.032192,0.532680,0.618182,5
2,main,financial_sporting_ridge,0.612861,0.113608,0.496599,0.740000,5
3,without_2020,baseline,0.663470,0.065405,0.600000,0.738095,4
4,without_2020,financial_ridge,0.576129,0.102219,0.472222,0.690476,4
5,without_2020,financial_sporting_ridge,0.596346,0.108499,0.482993,0.731481,4
